In [1]:
import argparse
import operator
import os
import re
from collections import defaultdict

from itertools import compress
try:
    from functools import reduce
except ImportError:  # python < 2
    pass

import tqdm

import numpy

import h5py

from gwdatafind.utils import filename_metadata

from ligo.segments import segmentlist
from ligo.segments.utils import fromsegwizard

from pycbc import __version__, events
from pycbc.inject import InjectionSet
from pycbc.io import FieldArray

from pycbc.events.hm_utils import TQDM_KW, read_hdf5_triggers, read_segment_files

In [2]:
def indices_from_eventids(net_ifo_eid, ifo_eid):
    # spot check that eventids correspond to the index of the array.
    # NB: This should already be the case, but just in case...
    if net_ifo_eid[-1] != ifo_eid[-1]:
        # if not, find the next matching index
        net_ifo_eid = [next(i for i, _ in enumerate(ifo_eid) 
                            if i == eid) for eid in net_ifo_eid]
    return net_ifo_eid


In [3]:
injf = "../inj/mini-bbh-3dets-512s_snr.hdf"
tmpinj = InjectionSet(injf).table
# trigfiles = ["./test9.hdf", "./test9.hdf"]
trigfiles = ["../inspiral/MINI-TESTING-512-realdata-INJECTIONS.hdf"]

In [4]:
triggers = read_hdf5_triggers(trigfiles)

In [50]:
exclude = read_segment_files([])
nexcluded = 0
missed = []
found = []
eids = []
num_trigs = []
# trigs = defaultdict(list)

allinjections = None


In [6]:
# ext = os.path.basename(injf)
# inj_is_xml = False
# if ext.endswith(('.xml', '.xml.gz', '.xmlgz')):
#     inj_is_xml = True

In [7]:
arg_dict = {'verbose': True, 'output': '/Users/camill/projects/pycbc/test_hm/test.hdf', 'instruments': ['H1', 'L1', 'V1'], 'bank_file': '/Users/camill/projects/pycbc/test_hm/bank/mini-bank.hdf', 'snr_threshold': 4.0, 'low_frequency_cutoff': 20.0, 'approximant': ['IMRPhenomXHM:mtotal<4', 'IMRPhenomXHM:else'], 'order': '-1', 'taper_template': None, 'cluster_method': 'window', 'cluster_window': 0.0, 'bank_veto_bank_file': None, 'downsample_factor': 1, 'upsample_threshold': None, 'upsample_method': 'pruned_fft', 'user_tag': None, 'coinc_threshold': 7.0, 'timing_error': 0.005, 'num_timeslides': 1, 'channel_name':  {'H1': 'H1:GWOSC-4KHZ_R1_STRAIN', 'L1': 'L1:GWOSC-4KHZ_R1_STRAIN', 'V1': 'V1:GWOSC-4KHZ_R1_STRAIN'}, 'frame_files':  {'H1': ['./strain/H-H1_GWOSC_O3b_4KHZ_R1-1267732480-4096.gwf'], 'L1': ['./strain/L-L1_GWOSC_O3b_4KHZ_R1-1267732480-4096.gwf'], 'V1': ['./strain/V-V1_GWOSC_O3b_4KHZ_R1-1267732480-4096.gwf']}, 'psdvar_segment': None, 'psdvar_short_segment': None, 'psdvar_long_segment': None, 'psdvar_psd_duration': None, 'psdvar_psd_stride': None, 'psdvar_low_freq': None, 'psdvar_high_freq': None, 'processing_scheme': 'cpu', 'processing_device_id': 0, 'fft_backends': [], 'fftw_measure_level': 0, 'fftw_threads_backend': None, 'fftw_input_float_wisdom_file': None, 'fftw_input_double_wisdom_file': None, 'fftw_output_float_wisdom_file': None, 'fftw_output_double_wisdom_file': None, 'fftw_import_system_wisdom': False, 'cpu_affinity': None, 'cpu_affinity_from_env': None, 'trig_start_time':None, 'trig_end_time':None}
arg_dict["sample_rate"] = defaultdict(lambda: 512)
arg_dict["gps_start_time"] = defaultdict(lambda: 1234)
arg_dict["gps_end_time"] = defaultdict(lambda: 1235)
arg_dict["segment_start_pad"] = defaultdict(lambda: 0)
arg_dict["segment_end_pad"] = defaultdict(lambda: 0)
arg_dict["rank_column"] = "snr_2_filter"
arg_dict["time_window"] = 1

class Args(object):
    def __init__(self, arg_dict):
        for k,v in arg_dict.items():
            setattr(self, k, v)
    def __getitem__(self, item):
        return getattr(self, item)
args = Args(arg_dict)

In [ ]:
def keep_ind(times, start, end):
    """ Return the list of indices within the list of start and end times
    """
    time_sorting = times.argsort()
    times = times[time_sorting]
    indices = numpy.array([], dtype=numpy.uint32)
    leftidx = numpy.searchsorted(times, start, side='left')
    rightidx = numpy.searchsorted(times, end, side='right')

    for li, ri in zip(leftidx, rightidx):
        seg_indices = numpy.arange(li, ri, 1).astype(numpy.uint32)
        indices=numpy.union1d(seg_indices, indices)
    return time_sorting[indices]

In [51]:
from ligo.lw import lsctables
import pycbc
from functools import singledispatch

@singledispatch
def take(injections, keep: numpy.ndarray):
    """Take values from injections where keep == True.
    """
    raise NotImplementedError("Please implement take.")

@take.register
def take_WaveformArray(
    injections: pycbc.io.record.WaveformArray, 
    keep: numpy.ndarray,
    ) -> pycbc.io.record.WaveformArray:
    return injections[keep]

@take.register
def take_SimInspiralTable(
    injections: lsctables.SimInspiralTable, 
    keep: numpy.ndarray,
    ) -> lsctables.SimInspiralTable:
    out = type(tmpinj)()
    out.extend(compress(injections, keep))
    return out

@singledispatch
def get_geocent_time(injections):
    raise NotImplementedError("Please implement get_geocent_time.")
    
@get_geocent_time.register
def get_geocent_time_WaveformArray(
    injections: pycbc.io.record.WaveformArray, 
    ) -> pycbc.io.record.WaveformArray:
    return injections["tc"]

@get_geocent_time.register
def get_geocent_time_SimInspiralTable(
    injections: lsctables.SimInspiralTable, 
    ) -> numpy.ndarray:
    return numpy.array([inj.time_geocent for inj in injections])


# read injections and filter by exclude segments
tmpinj = InjectionSet(injf).table
injtime = get_geocent_time(tmpinj)
keep = numpy.asarray([t not in exclude for t in injtime])
nexcluded += (~keep).sum()
injections = take(tmpinj, keep)
injtime = injtime[keep]

# read triggers
triggers = read_hdf5_triggers(trigfiles)

# use the mean ifo time to approximate geocent time. 
ifo_list = [k for k in triggers.keys() if k != "network"]
ifo_times = {}
for ifo in ifo_list:
    net_ifo_eid = triggers["network"]["{}_event_id".format(ifo)]
    ifo_eid = triggers[ifo]["event_id"]
    # eventids should correspond to indices, but just in case..
    net_ifo_eid = indices_from_eventids(net_ifo_eid, ifo_eid)
    ifo_times[ifo] = triggers[ifo]["end_time"][net_ifo_eid]
time = numpy.asarray([
    events.mean_if_greater_than_zero(v)[0] 
        for v in numpy.array(list(ifo_times.values())).T])

snr = triggers["network"][args.rank_column]
event_id = triggers["network"]["event_id"]
time_sorting = time.argsort()

# determine found or missed
_left = numpy.searchsorted(
    time[time_sorting],
    injtime - args.time_window,
    side='left',
)
_right = numpy.searchsorted(
    time[time_sorting],
    injtime + args.time_window,
    side='right',
)

# get indices of found/missed injections and finding trigger
for i, (l, r) in enumerate(
        zip(_left, _right),
        start=len(allinjections or []),
):
    n = r - l
    num_trigs.append(n)
    if n == 0:
        missed.append(i)
    else:
        found.append(i)
        # r-l == 1 means the injection has exactly one event 
        # associated, while r-l>1 indicates more than one
        # was associated within the window.
        #FIXME: below line isn't used; remove?
        eid = l if n == 1 else event_id[l + snr[l:r].argmax()]
        eids.append(eid)


After the loop, where all_injections updates perform the below step. Will need to append so might want to make it a list rather than a array. Then also add in the info about the number of trigs to the found colummn. 

In [58]:
trigs = {}
for col in triggers["network"]:
    trigs[col] = triggers["network"][col][eids]
    
for ifo in ifo_list:
    net_ifo_eid = triggers["network"]["{}_event_id".format(ifo)][eids]
    ifo_eid = triggers[ifo]["event_id"]
    # eventids should correspond to indices, but just in case..
    net_ifo_eid = indices_from_eventids(net_ifo_eid, ifo_eid)
    for col in triggers[ifo]:
        if "/" not in col and "event_id" not in col:
            trigs["{}_{}".format(ifo, col)] = triggers[ifo][col][net_ifo_eid]

In [59]:
trigs

{'H1_event_id': array([   48,  2068,  8559, 11038, 12535]),
 'L1_event_id': array([   48,  2068,  8559, 11038, 12535]),
 'V1_event_id': array([   48,  2068,  8559, 11038, 12535]),
 'event_id': array([   48,  2068,  8559, 11038, 12535]),
 'nifo': array([3, 3, 3, 3, 3]),
 'snr_2_filter': array([134.72842 , 138.63338 , 114.559586,  31.344652, 100.85781 ],
       dtype=float32),
 'snr_2_filter_rss': array([134.76082 , 138.66986 , 114.642685,  31.474512, 100.9232  ],
       dtype=float32),
 'template_id': array([ 0, 10, 42, 54, 62]),
 'timeslide_id': array([0, 0, 0, 0, 0]),
 'H1_coa_phase_dom': array([0., 0., 0., 0., 0.], dtype=float32),
 'H1_coa_phase_sub': array([0., 0., 0., 0., 0.], dtype=float32),
 'H1_end_time': array([1.26773605e+09, 1.26773605e+09, 1.26773605e+09, 1.26773622e+09,
        1.26773605e+09]),
 'H1_sigmasq_dom': array([1.3743028e+07, 1.0261772e+08, 1.8360498e+07, 2.4386598e+08,
        1.8403112e+08], dtype=float32),
 'H1_sigmasq_sub': array([15088000., 40240356., 2263553